In [1]:
pip install kafka-python

Note: you may need to restart the kernel to use updated packages.


# Producer 

In [1]:
import pandas as pd
import json
import time
import random
from kafka import KafkaProducer

# ==========================================
# 1. CONFIGURACIÓN DEL SIMULADOR
# ==========================================
KAFKA_BROKER = 'kafka:9093'  # El puerto interno de tu red Docker
KAFKA_TOPIC = 'transactions'

# CSV de Kaggle
CSV_PATH = '/home/jovyan/work/data/creditcard.csv' 

# ==========================================
# 2. INICIALIZAR EL PRODUCTOR KAFKA
# ==========================================
print(f"Conectando al broker Kafka en {KAFKA_BROKER}...")
producer = KafkaProducer(
    bootstrap_servers=[KAFKA_BROKER],
    # Serializamos automáticamente el diccionario de Python a JSON binario
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
print("¡Conexión establecida!")

# ==========================================
# 3. CARGA Y PREPARACIÓN DE DATOS
# ==========================================
print(f"Cargando dataset desde {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)

# Separamos los datos para poder manipular la probabilidad de inyección
df_fraud = df[df['Class'] == 1]
df_normal = df[df['Class'] == 0]

print(f"Dataset cargado. Fraudes totales: {len(df_fraud)} | Transacciones normales: {len(df_normal)}")
print("Iniciando inyección de datos en streaming (Presiona 'Stop' o Ctrl+C para detener)...")
print("-" * 50)

# ==========================================
# 4. BUCLE DE SIMULACIÓN (STREAMING)
# ==========================================
try:
    while True:
        if random.random() < 0.2 and not df_fraud.empty:
            # Escogemos una fila aleatoria de los fraudes
            row = df_fraud.sample(1).iloc[0]
            tx_type = "FRAUDE"
        else:
            # Escogemos una fila aleatoria de las normales
            row = df_normal.sample(1).iloc[0]
            tx_type = "NORMAL"

        # Convertimos la fila a diccionario y eliminamos la columna 'Class'
        payload = row.drop('Class').to_dict()

        # Enviamos el JSON por Kafka
        producer.send(KAFKA_TOPIC, value=payload)
        
        # Forzamos el envío inmediato
        producer.flush()

        # Imprimimos por pantalla para que veas qué está saliendo
        print(f"Enviado {tx_type} | Time: {payload.get('Time', 0):.0f} | Amount: ${payload.get('Amount', 0):.2f}")
        
        # Esperamos 1 segundo entre transacciones para simular tráfico real
        time.sleep(0.2)

except KeyboardInterrupt:
    print("\nSimulación detenida por el usuario.")
except Exception as e:
    print(f"\nError en el simulador: {e}")
finally:
    producer.close()
    print("Conexión con Kafka cerrada.")

Conectando al broker Kafka en kafka:9093...
¡Conexión establecida!
Cargando dataset desde /home/jovyan/work/data/creditcard.csv...
Dataset cargado. Fraudes totales: 492 | Transacciones normales: 284315
Iniciando inyección de datos en streaming (Presiona 'Stop' o Ctrl+C para detener)...
--------------------------------------------------
Enviado FRAUDE | Time: 94625 | Amount: $33.76
Enviado NORMAL | Time: 137705 | Amount: $20.42
Enviado NORMAL | Time: 53062 | Amount: $0.89
Enviado NORMAL | Time: 117110 | Amount: $15.03
Enviado FRAUDE | Time: 48380 | Amount: $208.58
Enviado NORMAL | Time: 147334 | Amount: $108.11
Enviado FRAUDE | Time: 68207 | Amount: $1.00
Enviado NORMAL | Time: 127860 | Amount: $8.00
Enviado NORMAL | Time: 123597 | Amount: $86.77
Enviado NORMAL | Time: 75861 | Amount: $2.58
Enviado NORMAL | Time: 93629 | Amount: $1.98
Enviado FRAUDE | Time: 93879 | Amount: $30.31
Enviado NORMAL | Time: 160875 | Amount: $5.00
Enviado NORMAL | Time: 41951 | Amount: $12.88
Enviado NORMAL |

## 1. Instalar el cliente de Kafka dentro de Jupyter
!pip install kafka-python-ng pandas

import time
import json
import pandas as pd
from kafka import KafkaProducer

